# Aula 4 — Testes Automatizados para Modelos de IA
## Testes estatísticos + Testes adversariais e avaliação de equidade (fairness)

IEC PUC Minas — Engenharia de Inteligência Artificial e MLOps

## Setup

In [ ]:
%pip install -q "ipytest==0.14.*" "numpy>=1.24"

import ipytest
import pytest
import numpy as np
ipytest.autoconfig()


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


---
## 1. Testes estatísticos

In [ ]:
%%ipytest

def accuracy(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return (y_true == y_pred).mean()


def test_accuracy_perfect_predictions():
    assert accuracy([1, 0, 1], [1, 0, 1]) == 1.0


def test_accuracy_all_wrong():
    assert accuracy([1, 0, 1], [0, 1, 0]) == 0.0


.

.

                                                                                           [100%]

2 passed in 0.98s


### Intervalo de confiança via bootstrap

In [ ]:
%%ipytest

def accuracy(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return (y_true == y_pred).mean()


def bootstrap_ci(y_true, y_pred, metric_fn=accuracy, n_boot=1000, ci=0.95, seed=42):
    rng = np.random.default_rng(seed)
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    n = len(y_true)
    boot_metrics = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        boot_metrics.append(metric_fn(y_true[idx], y_pred[idx]))
    lower = np.percentile(boot_metrics, (1 - ci) / 2 * 100)
    upper = np.percentile(boot_metrics, (1 + ci) / 2 * 100)
    return lower, upper


# dados fixos (hardcoded), não gerados por rng -- imunes a qualquer
# diferença de versão do numpy no ambiente onde rodar
y_true = [1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          0, 1, 1, 0, 0, 1, 1, 0, 1, 1]
y_pred_good = [1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1,
               1, 1, 1, 0, 0, 1, 0, 0, 1, 1]  # 6 erros plantados -> 80%


def test_bootstrap_ci_contains_point_estimate():
    point = accuracy(y_true, y_pred_good)
    lower, upper = bootstrap_ci(y_true, y_pred_good)
    assert lower <= point <= upper


def test_bootstrap_ci_of_perfect_predictions_is_a_single_point():
    lower, upper = bootstrap_ci([1, 1, 1, 1, 1], [1, 1, 1, 1, 1])
    assert lower == upper == 1.0


def test_bootstrap_ci_lower_never_exceeds_upper():
    lower, upper = bootstrap_ci(y_true, y_pred_good)
    assert lower <= upper


print(f"Acurácia pontual: {accuracy(y_true, y_pred_good):.2f}")
print(f"Intervalo de confiança (95%): {bootstrap_ci(y_true, y_pred_good)}")


Acurácia pontual: 0.80
Intervalo de confiança (95%): (np.float64(0.6666666666666666), np.float64(0.9333333333333333))


.

.

.

                                                                                          [100%]


3 passed in 0.15s


### Comparando dois modelos estatisticamente

In [ ]:
%%ipytest

def accuracy(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return (y_true == y_pred).mean()


def compare_models(y_true, pred_a, pred_b, n_boot=1000, ci=0.95, seed=42):
    rng = np.random.default_rng(seed)
    y_true = np.array(y_true)
    pred_a = np.array(pred_a)
    pred_b = np.array(pred_b)
    n = len(y_true)
    diffs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        diffs.append(accuracy(y_true[idx], pred_a[idx]) - accuracy(y_true[idx], pred_b[idx]))
    lo = np.percentile(diffs, (1 - ci) / 2 * 100)
    hi = np.percentile(diffs, (1 + ci) / 2 * 100)
    return lo, hi


y_true = [1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          0, 1, 1, 0, 0, 1, 1, 0, 1, 1]
y_pred_good = [1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1,
               1, 1, 1, 0, 0, 1, 0, 0, 1, 1]  # 80%
y_pred_bad = [1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0,
              0, 0, 1, 1, 0, 1, 0, 1, 0, 1]  # 50%
y_pred_similar = [1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0,
                  0, 1, 1, 1, 0, 1, 0, 0, 1, 1]  # ~77%


def test_clearly_better_model_is_significant():
    lo, hi = compare_models(y_true, y_pred_good, y_pred_bad)
    assert lo > 0 or hi < 0  # o intervalo NÃO inclui zero


def test_similar_models_are_not_significant():
    lo, hi = compare_models(y_true, y_pred_good, y_pred_similar)
    assert lo <= 0 <= hi  # o intervalo INCLUI zero


print(f"good={accuracy(y_true, y_pred_good):.2f}  bad={accuracy(y_true, y_pred_bad):.2f}  "
      f"similar={accuracy(y_true, y_pred_similar):.2f}")
print(f"diff good-bad (CI): {compare_models(y_true, y_pred_good, y_pred_bad)}")
print(f"diff good-similar (CI): {compare_models(y_true, y_pred_good, y_pred_similar)}")


good=0.80  bad=0.50  similar=0.77
diff good-bad (CI): (np.float64(0.09999999999999998), np.float64(0.5))
diff good-similar (CI): (np.float64(-0.16666666666666674), np.float64(0.2333333333333334))


.

.

                                                                                           [100%]


2 passed in 0.16s


---
## 2. Testes adversariais

Testamos se o modelo é **robusto** a mudanças que não deveriam importar (uma frase irrelevante no meio do texto) — mas continua sensível a mudanças que deveriam importar (o sentimento real mudou).

### Caso real: a placa de PARE adversarial (2018)

*Eykholt et al., "Robust Physical-World Attacks on Deep Learning Visual Classification", CVPR 2018.*

In [ ]:
%%ipytest

def mock_sentiment_model(text):
    text = text.lower()
    if "excelente" in text or "ótimo" in text or "bom" in text:
        return "positive"
    if "péssimo" in text or "horrível" in text or "ruim" in text:
        return "negative"
    return "neutral"


def test_adversarial_irrelevant_suffix_does_not_flip_prediction():
    base = "Este produto é excelente."
    perturbed = base + " Comprei na terça-feira."  # informação irrelevante
    # asserta o rótulo esperado explicitamente (não só igualdade entre os dois) --
    # senão um modelo quebrado que sempre devolve "neutral" também passaria
    assert mock_sentiment_model(base) == "positive"
    assert mock_sentiment_model(perturbed) == "positive"


def test_real_sentiment_change_does_flip_prediction():
    positive_text = "Este produto é excelente."
    negative_text = "Este produto é péssimo."
    # teste adversarial NÃO é sobre nunca mudar de opinião --
    # é sobre não mudar por motivo ERRADO. Uma mudança real de sentimento DEVE mudar a predição.
    assert mock_sentiment_model(positive_text) == "positive"
    assert mock_sentiment_model(negative_text) == "negative"


.

.

                                                                                           [100%]


2 passed in 0.03s


---
## 3. Fairness — paridade demográfica

### Caso real: COMPAS / ProPublica (2016)

O COMPAS era usado por tribunais nos EUA para estimar risco de reincidência, informando decisões de fiança e de pena.

### A matriz de confusão

Pra separar o que a ProPublica mediu (taxa de falso positivo) do que a Northpointe mediu (valor preditivo positivo), precisamos abrir a acurácia numa tabela 2x2:

|  | Previsto positivo | Previsto negativo |
|---|---|---|
| **Real positivo** | Verdadeiro Positivo (TP) | Falso Negativo (FN) |
| **Real negativo** | Falso Positivo (FP) | Verdadeiro Negativo (TN) |

- **FPR** (taxa de falso positivo) = FP / (FP + TN) — o que a ProPublica mediu por grupo
- **PPV** (valor preditivo positivo) = TP / (TP + FP) — o que a Northpointe mediu
- Acurácia = (TP + TN) / total — a métrica que já usamos na seção 1


In [ ]:
%%ipytest

def confusion_counts(y_true, y_pred):
    """TP, FP, FN, TN a partir de rótulos binários (0/1)."""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return {
        "TP": int(((y_true == 1) & (y_pred == 1)).sum()),
        "FP": int(((y_true == 0) & (y_pred == 1)).sum()),
        "FN": int(((y_true == 1) & (y_pred == 0)).sum()),
        "TN": int(((y_true == 0) & (y_pred == 0)).sum()),
    }


def false_positive_rate(y_true, y_pred):
    c = confusion_counts(y_true, y_pred)
    return c["FP"] / (c["FP"] + c["TN"])


def positive_predictive_value(y_true, y_pred):
    c = confusion_counts(y_true, y_pred)
    return c["TP"] / (c["TP"] + c["FP"])


# Réu rotulado "alto risco" = y_pred 1; reincidiu de fato = y_true 1.
# Números fictícios (não são os dados reais do COMPAS) -- só pra tornar
# a discussão tangível. Mesma taxa-base (3 em 10) nos dois grupos.
y_true_grupoA = [1, 1, 1, 0, 0, 0, 0, 0, 0, 0]
y_pred_grupoA = [1, 1, 0, 1, 1, 1, 0, 0, 0, 0]

y_true_grupoB = [1, 1, 1, 0, 0, 0, 0, 0, 0, 0]
y_pred_grupoB = [1, 1, 1, 1, 0, 0, 0, 0, 0, 0]


def test_mesma_taxa_base_nos_dois_grupos():
    assert sum(y_true_grupoA) == sum(y_true_grupoB)


def test_fpr_do_grupo_a_e_maior_que_o_grupo_b():
    # mesma taxa-base, mas o grupo A recebe mais falso positivo -- a queixa da ProPublica
    assert false_positive_rate(y_true_grupoA, y_pred_grupoA) > false_positive_rate(y_true_grupoB, y_pred_grupoB)


for nome, yt, yp in [("A", y_true_grupoA, y_pred_grupoA), ("B", y_true_grupoB, y_pred_grupoB)]:
    c = confusion_counts(yt, yp)
    print(f"Grupo {nome}: {c}  FPR={false_positive_rate(yt, yp):.2f}  PPV={positive_predictive_value(yt, yp):.2f}")


In [ ]:
%%ipytest

def positive_rate(y_pred, group):
    """Taxa de predições positivas por grupo. Espera y_pred numérico (0/1)."""
    y_pred_arr = np.array(y_pred)
    group_arr = np.array(group)
    return {g: y_pred_arr[group_arr == g].mean() for g in sorted(set(group))}


def check_demographic_parity(y_pred, group, max_gap=0.1):
    """Verifica paridade demográfica. Espera y_pred numérico e binário (0/1) --
    não rótulos de texto como "positive"/"negative" (ex: saída do mock_sentiment_model)."""
    y_pred_arr = np.array(y_pred)
    if y_pred_arr.dtype.kind not in "iuf":
        raise TypeError("check_demographic_parity espera y_pred numérico (0/1), não rótulos de texto")
    rates = positive_rate(y_pred, group)
    gap = max(rates.values()) - min(rates.values())
    if gap > max_gap:
        raise ValueError(f"Gap de paridade demográfica {gap:.2f} > {max_gap} entre grupos {rates}")
    return True


def test_check_demographic_parity_accepts_fair_predictions():
    group = ["A"] * 5 + ["B"] * 5
    y_pred_fair = [1, 0, 1, 0, 1] + [1, 0, 1, 0, 1]  # 0.6 em cada grupo
    assert check_demographic_parity(y_pred_fair, group) is True


def test_check_demographic_parity_rejects_unfair_predictions():
    group = ["A"] * 5 + ["B"] * 5
    y_pred_unfair = [1, 1, 1, 1, 0] + [0, 0, 0, 0, 1]  # 0.8 vs 0.2
    with pytest.raises(ValueError, match="paridade demográfica"):
        check_demographic_parity(y_pred_unfair, group)


def test_check_demographic_parity_rejects_text_labels():
    with pytest.raises(TypeError, match="numérico"):
        check_demographic_parity(["positive", "negative"], ["A", "B"])


group_demo = ["A"] * 5 + ["B"] * 5
print("Taxas (caso justo):", positive_rate([1, 0, 1, 0, 1, 1, 0, 1, 0, 1], group_demo))
print("Taxas (caso injusto):", positive_rate([1, 1, 1, 1, 0, 0, 0, 0, 0, 1], group_demo))


Taxas (caso justo): {'A': np.float64(0.6), 'B': np.float64(0.6)}
Taxas (caso injusto): {'A': np.float64(0.8), 'B': np.float64(0.2)}


.

.

.

                                                                                          [100%]


3 passed in 0.04s


---
## 4. Exercício

`model_x` tem 80% de acurácia (bem melhor que `model_y`, com 50%). Mas será que `model_x` trata os grupos A e B da mesma forma? Complete os dois testes abaixo.

In [ ]:
%%ipytest

def accuracy(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return (y_true == y_pred).mean()


def compare_models(y_true, pred_a, pred_b, n_boot=1000, ci=0.95, seed=42):
    rng = np.random.default_rng(seed)
    y_true = np.array(y_true)
    pred_a = np.array(pred_a)
    pred_b = np.array(pred_b)
    n = len(y_true)
    diffs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        diffs.append(accuracy(y_true[idx], pred_a[idx]) - accuracy(y_true[idx], pred_b[idx]))
    lo = np.percentile(diffs, (1 - ci) / 2 * 100)
    hi = np.percentile(diffs, (1 + ci) / 2 * 100)
    return lo, hi


def positive_rate(y_pred, group):
    y_pred_arr = np.array(y_pred)
    group_arr = np.array(group)
    return {g: y_pred_arr[group_arr == g].mean() for g in sorted(set(group))}


def check_demographic_parity(y_pred, group, max_gap=0.1):
    rates = positive_rate(y_pred, group)
    gap = max(rates.values()) - min(rates.values())
    if gap > max_gap:
        raise ValueError(f"Gap de paridade demográfica {gap:.2f} > {max_gap} entre grupos {rates}")
    return True


# 40 amostras (não 20) -- com amostra pequena, a diferença de significância
# pode variar por sorte da reamostragem; com 40, o resultado é estável
# (testado em 200 seeds diferentes, sem nenhuma falha)
y_true_ex = [1, 0] * 20
model_x_pred = [1, 0] * 16 + [0, 1] * 4          # acurácia 0.80
model_y_pred = [1] * 20 + [0] * 20               # acurácia 0.50
group_ex = ["A" if i % 2 == 0 else "B" for i in range(40)]


# TODO: use compare_models pra comprovar que model_x é significativamente melhor que model_y
def test_model_x_is_significantly_better_than_model_y():
    pytest.fail("TODO: calcule o intervalo com compare_models e comprove que exclui zero")


# TODO: use pytest.raises pra comprovar que model_x viola a paridade demográfica entre os grupos
def test_model_x_violates_demographic_parity():
    pytest.fail("TODO: escreva o teste com pytest.raises")


F

F

                                                                                           [100%]


============================================ FAILURES =============================================
________________________ test_model_x_is_significantly_better_than_model_y ________________________

    def test_model_x_is_significantly_better_than_model_y():
>       pytest.fail("TODO: calcule o intervalo com compare_models e comprove que exclui zero")
E       Failed: TODO: calcule o intervalo com compare_models e comprove que exclui zero

C:\Users\FHPA\AppData\Local\Temp\ipykernel_83116\1796693074.py:47: Failed
____________________________ test_model_x_violates_demographic_parity _____________________________

    def test_model_x_violates_demographic_parity():
>       pytest.fail("TODO: escreva o teste com pytest.raises")
E       Failed: TODO: escreva o teste com pytest.raises

C:\Users\FHPA\AppData\Local\Temp\ipykernel_83116\1796693074.py:52: Failed
===================================== short test summary info =====================================
FAILED t_4b6a784db37644128e44014

---
## 5. Suíte completa

In [ ]:
%%ipytest -v

# --- código sob teste -------------------------------------------------

def accuracy(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return (y_true == y_pred).mean()


def bootstrap_ci(y_true, y_pred, metric_fn=accuracy, n_boot=1000, ci=0.95, seed=42):
    rng = np.random.default_rng(seed)
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    n = len(y_true)
    boot_metrics = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        boot_metrics.append(metric_fn(y_true[idx], y_pred[idx]))
    lower = np.percentile(boot_metrics, (1 - ci) / 2 * 100)
    upper = np.percentile(boot_metrics, (1 + ci) / 2 * 100)
    return lower, upper


def compare_models(y_true, pred_a, pred_b, n_boot=1000, ci=0.95, seed=42):
    rng = np.random.default_rng(seed)
    y_true = np.array(y_true)
    pred_a = np.array(pred_a)
    pred_b = np.array(pred_b)
    n = len(y_true)
    diffs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        diffs.append(accuracy(y_true[idx], pred_a[idx]) - accuracy(y_true[idx], pred_b[idx]))
    lo = np.percentile(diffs, (1 - ci) / 2 * 100)
    hi = np.percentile(diffs, (1 + ci) / 2 * 100)
    return lo, hi


def mock_sentiment_model(text):
    text = text.lower()
    if "excelente" in text or "ótimo" in text or "bom" in text:
        return "positive"
    if "péssimo" in text or "horrível" in text or "ruim" in text:
        return "negative"
    return "neutral"


def positive_rate(y_pred, group):
    y_pred_arr = np.array(y_pred)
    group_arr = np.array(group)
    return {g: y_pred_arr[group_arr == g].mean() for g in sorted(set(group))}


def check_demographic_parity(y_pred, group, max_gap=0.1):
    rates = positive_rate(y_pred, group)
    gap = max(rates.values()) - min(rates.values())
    if gap > max_gap:
        raise ValueError(f"Gap de paridade demográfica {gap:.2f} > {max_gap} entre grupos {rates}")
    return True


# --- suíte de testes ----------------------------------------------------

def test_accuracy_perfect_predictions():
    assert accuracy([1, 0, 1], [1, 0, 1]) == 1.0


def test_bootstrap_ci_of_perfect_predictions_is_a_single_point():
    lower, upper = bootstrap_ci([1, 1, 1, 1, 1], [1, 1, 1, 1, 1])
    assert lower == upper == 1.0


def test_models_with_clear_gap_are_significant():
    y_true = [1, 0] * 20
    good = [1, 0] * 16 + [0, 1] * 4
    bad = [0, 1] * 20
    lo, hi = compare_models(y_true, good, bad)
    assert lo > 0 or hi < 0


def test_adversarial_irrelevant_suffix_does_not_flip_prediction():
    base = "Este produto é excelente."
    perturbed = base + " Comprei na terça-feira."
    assert mock_sentiment_model(base) == mock_sentiment_model(perturbed) == "positive"


def test_check_demographic_parity_rejects_unfair_predictions():
    group = ["A"] * 5 + ["B"] * 5
    y_pred_unfair = [1, 1, 1, 1, 0] + [0, 0, 0, 0, 1]
    with pytest.raises(ValueError, match="paridade demográfica"):
        check_demographic_parity(y_pred_unfair, group)


======================================= test session starts =======================================
platform win32 -- Python 3.13.9, pytest-9.0.3, pluggy-1.6.0
rootdir: C:\Users\FHPA\Documents\PUC Minas\Testes Automatizados IA\aula-4-27-08
plugins: anyio-4.11.0, Faker-37.11.0, hypothesis-6.164.0, langsmith-0.4.37, asyncio-1.3.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 5 items

t_4b6a784db37644128e4401421f5a83bf.py 

.

.

.

.

.

                                                  [100%]



======================================== 5 passed in 0.16s ========================================


---
## 6. [Prático] A métrica decide o veredito

In [ ]:
%%ipytest

import numpy as np

y_true_ex = [1, 0] * 20
model_x_pred = [1, 0] * 16 + [0, 1] * 4
group_ex = ["A" if i % 2 == 0 else "B" for i in range(40)]


def positive_rate(y_pred, group):
    y_pred_arr = np.array(y_pred)
    group_arr = np.array(group)
    return {g: y_pred_arr[group_arr == g].mean() for g in sorted(set(group))}


def check_demographic_parity(y_pred, group, max_gap=0.1):
    rates = positive_rate(y_pred, group)
    gap = max(rates.values()) - min(rates.values())
    if gap > max_gap:
        raise ValueError(f"Gap de paridade demográfica {gap:.2f} > {max_gap} entre grupos {rates}")
    return True


def check_accuracy_parity(y_true, y_pred, group, max_gap=0.1):
    """A acurácia deve ser parecida entre os grupos.

    TODO: calcular a acurácia dentro de cada grupo e levantar
    ValueError se o gap entre a maior e a menor passar de max_gap.
    """
    raise NotImplementedError("implementem check_accuracy_parity")


def test_model_x_passa_em_paridade_de_acuracia():
    assert check_accuracy_parity(y_true_ex, model_x_pred, group_ex) is True


def test_model_x_reprova_em_paridade_demografica():
    # o mesmo modelo, a outra métrica -- este ja passa desde o inicio
    with pytest.raises(ValueError, match="paridade demográfica"):
        check_demographic_parity(model_x_pred, group_ex)


F

.

                                                                                           [100%]


============================================ FAILURES =============================================
___________________________ test_model_x_passa_em_paridade_de_acuracia ____________________________

    def test_model_x_passa_em_paridade_de_acuracia():
>       assert check_accuracy_parity(y_true_ex, model_x_pred, group_ex) is True
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

C:\Users\FHPA\AppData\Local\Temp\ipykernel_83116\1713376694.py:32: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _

y_true = [1, 0, 1, 0, 1, 0, ...], y_pred = [1, 0, 1, 0, 1, 0, ...]
group = ['A', 'B', 'A', 'B', 'A', 'B', ...], max_gap = 0.1

    def check_accuracy_parity(y_true, y_pred, group, max_gap=0.1):
        """A acurácia deve ser parecida entre os grupos.
    
        TODO: calcular a acurácia dentro de cada grupo e levantar
        ValueError se o gap entre a maior e a menor passar de max_gap.
        """
>       raise Not